In [1]:
from utils import logger
from pipeline.pdf_parser import GrobidPDFParser
from pipeline.sentence_extractor import extract_sentences
from pipeline.reference_resolver import ReferenceResolver

import json


In [2]:
logger.info("Starting the application...")

logger.info("Parsing the PDF and extracting information...")

parser = GrobidPDFParser(pdf_path="../papers/BERT.pdf")
parsed_paper = parser.parse()

logger.info("Successfully parsed the PDF. Extracted information:")
logger.info(f"Title: {parsed_paper.title}")
logger.info(f"Abstract: {parsed_paper.abstract}")

2026-05-09 12:56:14,577 - missing_citations - INFO - Starting the application...
2026-05-09 12:56:14,579 - missing_citations - INFO - Parsing the PDF and extracting information...
2026-05-09 12:56:20,236 - missing_citations - INFO - Successfully parsed the PDF. Extracted information:
2026-05-09 12:56:20,238 - missing_citations - INFO - Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
2026-05-09 12:56:20,239 - missing_citations - INFO - Abstract: We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; BERT is conceptually simple and empirically powerful. It obtains new state-of-the-art results on eleven natural language processing tasks, including pushing the GLUE score to 80.5% (7.7% point absolute improvement), MultiNLI accuracy to 86.7% (4.6% absolute improvement), SQuAD v1.1 question answering Test 

In [3]:
logger.info("Extracting sentences from the parsed paper...")
sentences = extract_sentences(parsed_paper)
logger.info(f"Extracted {len(sentences)} sentences from the paper.")


2026-05-09 12:56:20,253 - missing_citations - INFO - Extracting sentences from the parsed paper...
2026-05-09 12:56:23,189 - missing_citations - INFO - Extracted 288 sentences from the paper.


In [4]:
logger.info("Resolving references in the paper...")
resolver = ReferenceResolver()
resolved_references = []

for ref in parsed_paper.references:
    resolved = resolver.resolve(ref)
    resolved_references.append(resolved)

logger.info("Resolved references:")
for ref, resolved in zip(parsed_paper.references, resolved_references):
    logger.info(f"Original: {ref}")
    logger.info(f"Resolved: {resolved}")
    print("---")

logger.info(f"Stats: {resolver.stats}")


2026-05-09 12:56:23,203 - missing_citations - INFO - Resolving references in the paper...
2026-05-09 12:57:12,047 - missing_citations - INFO - Resolved references:
2026-05-09 12:57:12,048 - missing_citations - INFO - Original: Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.
2026-05-09 12:57:12,049 - missing_citations - INFO - Resolved: ResolvedReference(raw_reference='Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.', resolved_paper_id='W2880875857', openalex_id=None, title='Contextual String Embeddings for Sequence Labeling', doi=None, method='fuzzy_title', confidence=0.95, unresolved_reason=None)
---
2026-05-09 12:57:12,050 - missing_citations - INFO - Original: Rami Al-Rfou, 

In [5]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from database.qdrant import create_qdrant_client
from utils.config import config

logger.info("Initializing models and Qdrant client...")

# Initialize Qdrant client
qdrant_client = create_qdrant_client(config.QDRANT_URL)
logger.info(f"Connected to Qdrant at {config.QDRANT_URL}")

# Load dense model
logger.info(f"Loading dense model: {config.DENSE_MODEL}...")
dense_model = SentenceTransformer(config.DENSE_MODEL)
logger.info("Dense model loaded")

# Load sparse model
logger.info(f"Loading sparse model: {config.SPARSE_MODEL}...")
sparse_model = SparseTextEmbedding(model_name=config.SPARSE_MODEL)
logger.info("Sparse model loaded")

c:\Users\sampe\OneDrive\Desktop\facultate\licenta\missing-citations-identifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-09 12:57:28,409 - missing_citations - INFO - Initializing models and Qdrant client...
2026-05-09 12:57:28,798 - missing_citations - INFO - Connected to Qdrant at http://localhost:6333
2026-05-09 12:57:28,800 - missing_citations - INFO - Loading dense model: intfloat/multilingual-e5-large-instruct...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3833.58it/s]


2026-05-09 12:57:40,589 - missing_citations - INFO - Dense model loaded
2026-05-09 12:57:40,591 - missing_citations - INFO - Loading sparse model: prithivida/Splade_PP_en_v1...
2026-05-09 12:57:42,098 - missing_citations - INFO - Sparse model loaded


In [6]:
from ingest_missing import ingest_openalex_papers
from indexer import EmbeddingIndex

# 1. Collect the IDs of all papers that were found externally
missing_ids = [
    ref.openalex_id 
    for ref in resolved_references 
    if ref.method == "openalex_external" and ref.openalex_id
]

if missing_ids:
    # 2. You will need to pass your initialized EmbeddingIndex. 
    # (Assuming you already have your qdrant_client, dense_model, etc. initialized)
    embedding_idx = EmbeddingIndex(
        qdrant_client=qdrant_client,
        dense_model=dense_model,
        sparse_model=sparse_model
    )
    
    # 3. Fetch, insert to Postgres, embed, and insert to Qdrant!
    inserted_count = ingest_openalex_papers(missing_ids, embedding_idx)
    print(f"Successfully ingested {inserted_count} missing papers into the local corpus.")


2026-05-09 12:57:43,225 - missing_citations - INFO - Inserted 1 new papers into PostgreSQL.
2026-05-09 12:57:44,989 - missing_citations - INFO - Upserted batch 1/1 (1 papers, running total=1).
2026-05-09 12:57:44,991 - missing_citations - INFO - Embedded and indexed 1 papers into Qdrant.
Successfully ingested 1 missing papers into the local corpus.
